# AOI and Soil Map Intersection

## GeoSoil-Cuba

This notebook prepares the soil dataset for the GeoSoil-Cuba analysis by
identifying and extracting the soil units that intersect the selected
Area of Interest (AOI) in Jaruco, Mayabeque, Cuba.

The project combines historical soil mapping with current satellite-derived
vegetation information. Therefore, this step establishes the spatial soil
framework that will later be compared with vegetation conditions derived
from Sentinel-2 imagery and NDVI.

### Objectives

1. Load the Mayabeque 1:25,000 soil map.
2. Load the selected AOI.
3. inspect and harmonize their coordinate reference systems (CRS).
4. Validate the soil geometries.
5. Identify soil units intersecting the AOI.
6. Clip the soil layer to the AOI.
7. Evaluate soil-map coverage within the AOI.
8. Inspect potential geometry and topology issues.
9. Prepare a processed soil layer for subsequent analysis.

### Data sources

- **Soil data:** Mayabeque 1:25,000 soil map in MapInfo format.
- **AOI:** Jaruco/Mayabeque study-area polygon in KML format.

The original soil data are preserved unchanged. Any geometry repairs,
transformations, or derived datasets are performed in memory or saved
separately as processed data.

### Analytical context

The final objective of GeoSoil-Cuba is not simply to map soil classes.
The soil units extracted here will later provide the spatial framework
for analyzing the relationship between soil characteristics and current
vegetation vigor using satellite-derived NDVI.

## 1. Project setup and source data

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt


# Project directories
PROJECT_DIR = Path("..").resolve()

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"


# Input datasets
SOIL_DIR = RAW_DIR / "Mapa_Suelos_25000"
SOIL_FILE = SOIL_DIR / "24 Myb suelos 25 mil.TAB"

AOI_FILE = RAW_DIR / "aoi" / "geosoil-cuba-jaruco.kml"

WATER_DIR = RAW_DIR / "Mapa_Suelos_25000"


# Processed-data directories
VECTOR_DIR = PROCESSED_DIR / "vector"
TABLE_DIR = PROCESSED_DIR / "tables"

VECTOR_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Soil dataset:", SOIL_FILE)
print("AOI:", AOI_FILE)
print("Water dataset directory:", WATER_DIR)

### 1.1 Load the spatial datasets

The Mayabeque soil map is provided in MapInfo format. GeoPandas can read
the dataset through its `.TAB` file, which references the associated
geometry and attribute files.

The AOI is provided as a KML file and is expected to use geographic
coordinates (longitude and latitude).

The first step is to load both layers and inspect their basic spatial
properties, including feature count, geometry type, bounds, and
coordinate reference system (CRS).

In [ ]:
# Load the Mayabeque soil map
soil_gdf = gpd.read_file(SOIL_FILE)

# Load the Area of Interest
aoi_gdf = gpd.read_file(AOI_FILE)

print("Soil features:", len(soil_gdf))
print("AOI features:", len(aoi_gdf))

In [ ]:
print("Soil CRS:")
print(soil_gdf.crs)

print("\nAOI CRS:")
print(aoi_gdf.crs)

### 1.2 Identify the soil-map CRS

The soil dataset contains a complete projected CRS definition in its
MapInfo metadata. GeoPandas displays the CRS using its Well-Known Text
(WKT) representation rather than a simple EPSG identifier.

The projection parameters correspond to **NAD27 / Cuba Norte (EPSG:3795)**.
We can ask the CRS library to identify the corresponding EPSG authority
code before using it in subsequent spatial operations.

In [ ]:
print("Soil CRS authority:")
print(soil_gdf.crs.to_authority())

In [ ]:
print("AOI CRS authority:")
print(aoi_gdf.crs.to_authority())

### 1.3 Harmonize the coordinate reference systems

The soil dataset contains a complete Lambert Conformal Conic projection
definition, but its CRS metadata does not resolve automatically to an EPSG
authority code.

Inspection of the projection parameters identifies it as
**NAD27 / Cuba Norte (EPSG:3795)**.

The AOI uses **WGS 84 (EPSG:4326)**.

For the spatial intersection, both layers must use the same CRS. We will
retain the original soil GeoDataFrame and create a separate analysis copy
with the identified EPSG authority explicitly assigned.

In [ ]:
# Preserve the original soil dataset
soil_analysis = soil_gdf.copy()

# Explicitly associate the identified CRS
soil_analysis = soil_analysis.set_crs(
    epsg=3795,
    allow_override=True
)

print("Soil analysis CRS:")
print(soil_analysis.crs)

print("\nSoil CRS authority:")
print(soil_analysis.crs.to_authority())

In [ ]:
print("Soil geometry types:")
print(soil_gdf.geometry.geom_type.value_counts())

print("\nSoil bounds:")
print(soil_gdf.total_bounds)

print("\nAOI bounds:")
print(aoi_gdf.total_bounds)

### 1.4 Reproject the AOI to the soil-map CRS

The soil map uses the projected CRS NAD27 / Cuba Norte (EPSG:3795),
while the AOI is stored in WGS 84 geographic coordinates (EPSG:4326).

Before performing any spatial operation, both datasets must use the same
coordinate reference system.

The soil map CRS will be used as the working CRS for this step. The AOI
coordinates are therefore transformed from EPSG:4326 to EPSG:3795.

The original AOI GeoDataFrame is preserved unchanged.

In [ ]:
# Reproject the AOI to the soil map's working CRS
aoi_projected = aoi_gdf.to_crs(soil_analysis.crs)

print("Original AOI CRS:")
print(aoi_gdf.crs)

print("\nProjected AOI CRS:")
print(aoi_projected.crs)

### 1.5 Check spatial alignment

Both datasets now use EPSG:3795. A simple overlay plot provides a visual
check that the AOI is positioned correctly relative to the Mayabeque soil
map.

This is a sanity check rather than the analytical intersection itself.

In [ ]:
# Plot AOI and Mayabeque soil map to verify CRS alignment

MAPS_DIR = PROJECT_DIR / "outputs" / "maps"
MAPS_DIR.mkdir(parents=True, exist_ok=True)

output_file = MAPS_DIR / "aoi_soil_crs_alignment.png"

fig, ax = plt.subplots(figsize=(12, 10))

soil_analysis.plot(
    ax=ax,
    edgecolor="black",
    linewidth=0.2,
    facecolor="none",
)

aoi_projected.boundary.plot(
    ax=ax,
    linewidth=2,
)

ax.set_title("AOI and Mayabeque Soil Map — CRS Alignment")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_aspect("equal")

plt.tight_layout()

fig.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Saved: {output_file}")

## 2. Geometry validation and repair

Before performing spatial operations, the soil polygons are checked for
geometric validity.

Invalid geometries can cause errors or unexpected results during operations
such as intersection, clipping, overlay, and area calculation.

The original soil dataset is preserved unchanged. Any repairs are performed
on the analysis copy (`soil_analysis`).

We first inspect the geometry validity without modifying the data.

In [ ]:

output_file = MAPS_DIR / "invalid_geometries.png"

# Identify invalid geometries (note the negation symbol '~' used to select only the false occurrences as True for the mask)
invalid_mask = ~soil_analysis.geometry.is_valid

fig, ax = plt.subplots(figsize=(12, 10))

# Plot all soil polygons
soil_analysis.plot(
    ax=ax,
    facecolor="none",
    edgecolor="lightgray",
    linewidth=0.5
)

# Plot invalid geometries on top
soil_analysis.loc[invalid_mask].plot(
    ax=ax,
    facecolor="red",
    edgecolor="black",
    linewidth=1.5
)


aoi_projected.boundary.plot(
    ax=ax,
    linewidth=2
)

ax.set_title("GeoSoil-Cuba — Soil Geometry Validation")
ax.set_axis_off()
ax.set_aspect("equal")

plt.tight_layout()

fig.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Saved: {output_file}")

In [ ]:
# Check geometry validity
soil_valid = soil_analysis.geometry.is_valid

print("Total soil features:", len(soil_analysis))
print("Valid geometries:", soil_valid.sum())
print("Invalid geometries:", (~soil_valid).sum())

In [ ]:
# Show the invalid geometries type and error coordinates
from shapely.validation import explain_validity

invalid_mask = ~soil_analysis.geometry.is_valid

invalid_reasons = soil_analysis.loc[
    invalid_mask, "geometry"
].apply(explain_validity)

invalid_reasons

### 2.1 Plot an invalid geometry as an example

The selected invalid geometry falls inside the AOI

In [ ]:
from shapely.geometry import Point

output_file = MAPS_DIR / "aoi_invalid_geometry.png"

feature = soil_analysis.loc[[1133]]

error_point = Point(394706.4, 345428)

fig, ax = plt.subplots(figsize=(8, 8))

feature.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=1.5
)

gpd.GeoSeries([error_point], crs=soil_analysis.crs).plot(
    ax=ax,
    color="red",
    markersize=80
)

ax.set_title("Feature 1133 — Ring Self-intersection")
ax.set_axis_off()

fig.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Saved: {output_file}")






### 2.2 Diagnose invalid geometries

Knowing that geometries are invalid is not sufficient. We also inspect the
reason for invalidity to determine whether the problem is minor and
repairable or requires further investigation. This output shows a better formatted table.

In [ ]:
invalid_reasons = (
    soil_analysis.loc[invalid_mask, ["ID", "geometry"]]
    .assign(invalid_reason=lambda df: df.geometry.apply(explain_validity))
    .drop(columns="geometry")
)

invalid_reasons

### 2.3 Repair invalid geometries

The invalid geometries are repaired using `make_valid()`.

This operation attempts to construct a geometrically valid representation
while preserving the spatial information as closely as possible.

The repaired geometries remain in the analysis copy and do not modify the
original MapInfo dataset.

In [ ]:
from shapely.validation import make_valid

soil_analysis.loc[invalid_mask, "geometry"] = (
    soil_analysis.loc[invalid_mask, "geometry"]
    .apply(make_valid)
)

In [ ]:
soil_valid_after = soil_analysis.geometry.is_valid

print("Valid geometries after repair:", soil_valid_after.sum())
print("Invalid geometries after repair:", (~soil_valid_after).sum())

In [ ]:
print(
    soil_analysis.geometry.geom_type.value_counts()
)

In [ ]:
# Inspect the contents of GeometryCollections

gc = soil_analysis[
    soil_analysis.geometry.geom_type == "GeometryCollection"
].copy()

print(f"GeometryCollections: {len(gc)}")

for idx, geom in gc.geometry.items():
    print(f"\nIndex: {idx}")
    print(f"Number of parts: {len(geom.geoms)}")
    
    for part in geom.geoms:
        print(f"  {part.geom_type}")

### 2.4 Normalize repaired geometries

We have confirmed that the nine GeometryCollections contain valid polygonal soil geometry plus LineString components. For our soil-area analysis, we want to keep the Polygon or MultiPolygon and exclude the linear component from the analysis geometry.

The following function handles three cases:

Existing Polygon → keep unchanged.
Existing MultiPolygon → keep unchanged.
GeometryCollection → extract its polygonal components and discard the non-area components.

We apply this only to soil_analysis, our working analysis layer; the original soil dataset remains untouched.

In [ ]:
from shapely.geometry import MultiPolygon

def extract_polygonal_geometry(geom):
    """
    Keep Polygon/MultiPolygon geometries.
    For GeometryCollections, extract only polygonal components.
    Non-area geometries such as LineString are excluded
    from the analysis geometry.
    """

    if geom.geom_type in ["Polygon", "MultiPolygon"]:
        return geom

    if geom.geom_type == "GeometryCollection":
        polygon_parts = [
            part
            for part in geom.geoms
            if part.geom_type in ["Polygon", "MultiPolygon"]
        ]

        if len(polygon_parts) == 1:
            return polygon_parts[0]

        polygons = []

        for part in polygon_parts:
            if part.geom_type == "Polygon":
                polygons.append(part)
            elif part.geom_type == "MultiPolygon":
                polygons.extend(list(part.geoms))

        if polygons:
            return MultiPolygon(polygons)

    return None


soil_analysis["geometry"] = soil_analysis.geometry.apply(
    extract_polygonal_geometry
)

### 2.5 Validate the repaired soil dataset

The soil analysis dataset has been validated after geometry repair and normalization. We confirmed that the expected number of soil features is preserved, no geometries are missing, the resulting geometries are topologically valid, and the geometry types are appropriate for polygon-based soil analysis.

After repairing the 10 invalid geometries, we perform a complete topology check.

The expected result is:

    1,201 total features
    0 null geometries
    0 invalid geometries

We also verify the final geometry types because make_valid() can legitimately change a geometry from Polygon to MultiPolygon or, in some cases, produce a GeometryCollection.

In [ ]:
# Post-normalization geometry validation

print(f"Total soil features: {len(soil_analysis)}")

print("\nGeometry types:")
print(soil_analysis.geometry.geom_type.value_counts())

null_geometries = soil_analysis.geometry.isna().sum()
print(f"\nNull geometries: {null_geometries}")

invalid_mask = ~soil_analysis.geometry.is_valid
invalid_count = invalid_mask.sum()

print(f"Invalid geometries: {invalid_count}")

if invalid_mask.any():
    print("\nInvalid feature IDs:")
    print(
        soil_analysis.loc[
            invalid_mask, "ID"
        ].tolist()
    )

## 3. AOI intersection, clipping, and soil coverage



### 3.1 Create the AOI geometry

The AOI is represented by a single geometry so that it can be used consistently in the spatial operations that follow.

The AOI and soil layer are both stored in EPSG:3795, so no reprojection is required for these operations.

In [ ]:
# Create a single geometry representing the complete AOI
aoi_geometry = aoi_projected.union_all()

print("AOI geometry created.")

### 3.2 Identify soil features intersecting the AOI

Before clipping the soil dataset to the Jaruco AOI, identify the soil features that intersect the AOI.

The intersects() operation determines whether a soil geometry overlaps or touches the AOI. It does not modify the soil geometries.

This produces a Boolean mask that will be used to create a smaller candidate dataset for the clipping operation.

In [ ]:
# Identify soil features that intersect the AOI
soil_intersects_aoi = soil_analysis.geometry.intersects(
    aoi_geometry
)

print("Soil features intersecting AOI:")
print(soil_intersects_aoi.sum())

print("\nSoil features NOT intersecting AOI:")
print((~soil_intersects_aoi).sum())

A total of 82 soil features intersect the Jaruco AOI, while 1,119 features do not.

The intersecting features form the candidate subset for the clipping operation.

### 3.3 Create the soil candidate subset

Use the intersection mask to extract only the soil features that intersect the AOI.

At this stage, the geometries are still unchanged. Features that extend beyond the AOI boundary are still represented by their original geometries.

The resulting subset will be used in the clipping operation.

In [ ]:
# Create the candidate soil subset
soil_aoi_candidates = soil_analysis[
    soil_intersects_aoi
].copy()

print(
    f"Candidate soil features: "
    f"{len(soil_aoi_candidates)}"
)

print("\nGeometry types:")
print(
    soil_aoi_candidates.geometry.geom_type.value_counts()
)

The candidate subset contains 82 features: 80 Polygon features and 2 MultiPolygon feature.

### 3.4 Clip the candidate soil features to the AOI

The candidate features are now clipped to the AOI boundary.

Unlike intersects(), the intersection() operation modifies the geometry by retaining only the portion that falls within the AOI.

Because both datasets use EPSG:3795, no reprojection is required before this operation.

In [ ]:
# Clip the candidate soil features to the AOI
soil_aoi_clipped = soil_aoi_candidates.copy()

soil_aoi_clipped["geometry"] = (
    soil_aoi_clipped.geometry.intersection(aoi_geometry)
)

print(f"Clipped soil features: {len(soil_aoi_clipped)}")

### 3.5 Validate the clipped soil geometries

Spatial clipping can potentially produce null, empty, or invalid geometries. Before calculating areas or performing further spatial analysis, verify the resulting geometries.

The clipped layer is expected to contain valid, non-empty geometries with no null geometry values.

In [ ]:
# Validate the clipped soil geometries

print(f"Clipped soil features: {len(soil_aoi_clipped)}")

print("\nNull geometries:")
print(soil_aoi_clipped.geometry.isna().sum())

print("\nEmpty geometries:")
print(soil_aoi_clipped.geometry.is_empty.sum())

print("\nInvalid geometries:")
print(
    (~soil_aoi_clipped.geometry.is_valid).sum()
)

print("\nGeometry types:")
print(
    soil_aoi_clipped.geometry.geom_type.value_counts()
)

If your actual result is zero for null, empty, and invalid:

The clipped soil layer contains no null, empty, or invalid geometries. The dataset is therefore suitable for the area calculations that follow.

### 3.6 Calculate the area of each clipped soil feature

The clipped soil geometries are stored in EPSG:3795, a projected coordinate reference system with metre-based coordinates. Therefore, the area property returns area in square metres.

Calculate the area of each individual soil feature and store it in square metres, hectares, and square kilometres.

These feature-level areas will be used later for the soil inventory and distribution analysis.

In [ ]:
# Calculate the area of each clipped soil feature

soil_aoi_clipped["area_m2"] = (
    soil_aoi_clipped.geometry.area
)

soil_aoi_clipped["area_ha"] = (
    soil_aoi_clipped["area_m2"] / 10_000
)

soil_aoi_clipped["area_km2"] = (
    soil_aoi_clipped["area_m2"] / 1_000_000
)

print("Area calculation completed.")

print("\nArea summary:")
print(
    soil_aoi_clipped[
        ["area_m2", "area_ha", "area_km2"]
    ].describe()
)

### 3.7 Calculate the spatial coverage of the soil layer

TThe area of individual soil features is useful for inventory calculations, but summing those areas does not necessarily represent the actual area covered by the soil layer because overlapping polygons would be counted more than once.

To measure the actual spatial coverage of the soil dataset within the AOI, dissolve the clipped soil geometries into a single union and calculate the area of that union.

The difference between the AOI area and the union of the soil geometries represents the area not covered by the soil dataset.

In [ ]:
# Calculate AOI area
aoi_area_m2 = aoi_geometry.area
aoi_area_ha = aoi_area_m2 / 10_000
aoi_area_km2 = aoi_area_m2 / 1_000_000

# Calculate the actual spatial coverage of the soil layer
soil_union = soil_aoi_clipped.geometry.union_all()

soil_coverage_m2 = soil_union.area
soil_coverage_ha = soil_coverage_m2 / 10_000
soil_coverage_km2 = soil_coverage_m2 / 1_000_000

# Calculate uncovered area
uncovered_area_km2 = (
    aoi_area_km2 - soil_coverage_km2
)

# Calculate percentage of the AOI covered by soil data
coverage_percentage = (
    soil_coverage_km2 / aoi_area_km2 * 100
)

print("AOI area:")
print(f"  {aoi_area_m2:,.2f} m²")
print(f"  {aoi_area_ha:,.2f} ha")
print(f"  {aoi_area_km2:,.2f} km²")

print("\nSoil spatial coverage:")
print(f"  {soil_coverage_m2:,.2f} m²")
print(f"  {soil_coverage_ha:,.2f} ha")
print(f"  {soil_coverage_km2:,.2f} km²")

print("\nUncovered AOI area:")
print(f"  {uncovered_area_km2:,.2f} km²")

print("\nSoil spatial coverage percentage:")
print(f"  {coverage_percentage:.2f}%")

The union of the clipped soil polygons represents the actual area covered by the soil dataset, with overlapping areas counted only once.

The soil layer covers **162.85 km²**, corresponding to **91.25% of the 178.47 km² AOI**. The remaining **15.62 km² (8.75%)** is not covered by the soil polygons.

However, this remaining area should **not yet be interpreted as missing soil data**. The AOI contains the Mampostón reservoir and other water bodies, which are not expected to have soil polygons. The complete water-body dataset will therefore be used to determine how much of this apparent gap is water and how much, if any, represents terrestrial area without soil coverage.

## 4. Water-body analysis

The GeoSoil-Cuba AOI contains the Mampostón reservoir, and the complete water-body dataset will allow us to determine whether the apparent soil-coverage gap corresponds to water rather than missing soil data.

The water dataset is stored as a MapInfo dataset in the raw-data directory. We first load and inspect the layer without modifying it. This allows us to verify its feature count, coordinate reference system, geometry types, and available attributes before incorporating it into the analysis.

In [ ]:
# Load the raw water-body dataset

WATER_FILE = WATER_DIR / "4 Cuerpos de agua.TAB"

water_gdf = gpd.read_file(WATER_FILE)

print("Water-body layer:")
print(f"  Features: {len(water_gdf):,}")
print(f"  CRS: {water_gdf.crs}")

print("\nGeometry types:")
print(water_gdf.geometry.geom_type.value_counts())

print("\nColumns:")
print(water_gdf.columns.tolist())

### 4.1 Inspect the water-body dataset

The water-body dataset contains **353 features**, including 337 polygons, one multipolygon, and 15 line features. The polygonal and multipolygon features can be used to calculate water-surface coverage, while the line features will be excluded from area calculations.

The dataset uses the same NAD27 / Cuba Lambert Conformal Conic coordinate system as the soil dataset and will therefore be standardized to **EPSG:3795**, our working CRS for GeoSoil-Cuba.

The attribute table includes a water-body name field (`Nombre`) and a recorded area field (`Area_km²`). These attributes will allow us to identify Mampostón and compare its recorded area with the area calculated directly from its geometry.

The next step is to standardize the CRS and identify the water features that intersect the Jaruco AOI.

In [ ]:
## Standardize the water-body CRS

water_analysis = water_gdf.copy()

# The source CRS parameters correspond to EPSG:3795.
# Assign the CRS explicitly without transforming coordinates.
water_analysis = water_analysis.set_crs(
    epsg=3795,
    allow_override=True
)

print("Water analysis CRS:")
print(water_analysis.crs)

print("\nCRS authority:")
print(water_analysis.crs.to_authority())

### 4.2 Identify relevant water features 

First we want to establish which features are relevant and inspect their attributes. Line features are retained at this stage because we're identifying intersecting features, but we'll exclude them later when calculating area.

In [ ]:
## Identify water bodies intersecting the Jaruco AOI

water_intersects_aoi = water_analysis.geometry.intersects(aoi_geometry)

water_aoi_candidates = water_analysis[
    water_intersects_aoi
].copy()

print("Water features intersecting AOI:")
print(f"  {len(water_aoi_candidates):,}")

print("\nGeometry types:")
print(
    water_aoi_candidates.geometry.geom_type.value_counts()
)

print("\nWater bodies by type:")
print(
    water_aoi_candidates["Tipo"].value_counts()
)

### 4.3 Clip relevant water features to the AOI

We now clip the intersecting reservoir to the AOI. This creates the analysis geometry representing only the portion of the reservoir that falls within our study area. We keep the original intersecting layer unchanged.

In [ ]:
# Reproject the AOI to the GeoSoil-Cuba working CRS

aoi_analysis = aoi_gdf.to_crs(epsg=3795)

# Clip the water body to the AOI

water_clipped = gpd.clip(
    water_aoi_candidates,
    aoi_analysis
)

print("Clipped water features:")
print(len(water_clipped))

print("\nGeometry types:")
print(water_clipped.geometry.geom_type.value_counts())

print("\nWater bodies by type:")
print(water_clipped["Tipo"].value_counts())

### 4.4 Calculate water-body area

Now we'll calculate the clipped reservoir's area in square meters and convert it to square kilometers. We'll also calculate what percentage of the entire AOI it occupies.

In [ ]:
# Calculate water-body and AOI areas

water_clipped["area_m2"] = water_clipped.geometry.area
water_clipped["area_km2"] = water_clipped["area_m2"] / 1_000_000

aoi_area_m2 = aoi_analysis.geometry.area.sum()
aoi_area_km2 = aoi_area_m2 / 1_000_000

water_area_km2 = water_clipped["area_km2"].sum()
water_percentage = (water_area_km2 / aoi_area_km2) * 100

print(f"AOI area: {aoi_area_km2:.2f} km²")
print(f"Water area: {water_area_km2:.2f} km²")
print(f"Water coverage: {water_percentage:.2f}%")

### 4.5 Validate the clipped water geometries

Before moving on, let's perform a final sanity check. We want to make sure the clipped geometry is valid and that no unexpected geometry types or empty geometries were introduced during clipping.

In [ ]:
# Verify the clipped water geometry

print("Empty geometries:")
print(water_clipped.geometry.is_empty.sum())

print("\nInvalid geometries:")
print((~water_clipped.geometry.is_valid).sum())

print("\nGeometry types:")
print(water_clipped.geometry.geom_type.value_counts())

print("\nCalculated area:")
print(water_clipped[["Tipo", "area_m2", "area_km2"]])

### 4.6 Visually inspect the clipped water bodies

The clipped water-body geometries are plotted over the study-area boundary
to visually verify their spatial extent and alignment with the AOI.

In [ ]:
# Plot clipped water bodies over the AOI

fig, ax = plt.subplots(figsize=(10, 8))

aoi_projected.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1.2,
    label="AOI",
)

water_clipped.plot(
    ax=ax,
    alpha=0.6,
    edgecolor="black",
    linewidth=0.5,
    label="Water bodies",
)

ax.set_title("Water Bodies Clipped to the AOI")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.legend()
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

In [ ]:
# Save the visual inspection figure

figure_path = OUTPUT_DIR / "figures" / "water_bodies_clipped_aoi.png"

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

print("Saved:", figure_path)

### 4.7 Create a compact water summary

Let's preserve the result in a small summary object rather than carrying unnecessary source attributes into later analysis.

In [ ]:
# Create a compact summary of water coverage

water_summary = {
    "aoi_area_km2": aoi_area_km2,
    "water_area_km2": water_area_km2,
    "water_coverage_pct": water_percentage
}

print("Water coverage summary:")
for key, value in water_summary.items():
    print(f"{key}: {value:.4f}")

## 5. Soil-data gaps and reservoir exclusion

In [ ]:
# Calculate the area of the AOI not covered by the clipped soil polygons

soil_coverage_geometry = soil_aoi_clipped.geometry.union_all()

soil_gap_geometry = aoi_geometry.difference(
    soil_coverage_geometry
)

soil_gap_area_m2 = soil_gap_geometry.area
soil_gap_area_km2 = soil_gap_area_m2 / 1_000_000

print(f"AOI area: {aoi_area_km2:.2f} km²")
print(f"Soil-covered area: "
      f"{soil_coverage_geometry.area / 1_000_000:.2f} km²")
print(f"Soil-data gap: {soil_gap_area_km2:.2f} km²")

### 5.1 Compare the soil-data gap with the Mampostón reservoir

In [ ]:
# Compare the soil-data gap with the clipped Mamposton reservoir

gap_water_intersection = soil_gap_geometry.intersection(
    water_clipped.geometry.union_all()
)

gap_water_area_km2 = gap_water_intersection.area / 1_000_000

gap_nonwater_geometry = soil_gap_geometry.difference(
    water_clipped.geometry.union_all()
)

gap_nonwater_area_km2 = gap_nonwater_geometry.area / 1_000_000

print(f"Soil-data gap: {soil_gap_area_km2:.2f} km²")
print(f"Gap explained by water: {gap_water_area_km2:.2f} km²")
print(f"Remaining non-water gap: {gap_nonwater_area_km2:.2f} km²")

### 5.2 Quantify the soil/reservoir overlap

Let's determine exactly how much of the reservoir is already covered by the soil dataset and how much is missing.

In [ ]:
# Compare the reservoir with the soil coverage

water_geometry = water_clipped.geometry.union_all()

water_soil_overlap = water_geometry.intersection(
    soil_coverage_geometry
)

water_already_covered_km2 = (
    water_soil_overlap.area / 1_000_000
)

water_missing_km2 = (
    water_area_km2 - water_already_covered_km2
)

print(f"Total reservoir area: "
      f"{water_area_km2:.2f} km²")

print(f"Reservoir already covered by soil polygons: "
      f"{water_already_covered_km2:.2f} km²")

print(f"Reservoir not covered by soil polygons: "
      f"{water_missing_km2:.2f} km²")

### 5.3 Create the missing reservoir geometry

We subtract the existing soil coverage from the reservoir. The result will be the exact portion of Mamposton that is absent from the soil dataset.

In [ ]:
# Create the portion of the reservoir missing from soil coverage

missing_water_geometry = water_geometry.difference(
    soil_coverage_geometry
)

missing_water_area_km2 = (
    missing_water_geometry.area / 1_000_000
)

print(f"Missing reservoir area: "
      f"{missing_water_area_km2:.2f} km²")

print("\nGeometry type:")
print(missing_water_geometry.geom_type)

### 5.4 Create the missing-water feature

In [ ]:
# Create a GeoDataFrame for the reservoir area missing from the soil dataset

missing_water = gpd.GeoDataFrame(
    {
        "land_cover": ["Water"],
        "water_type": ["Embalse"],
        "area_km2": [missing_water_area_km2]
    },
    geometry=[missing_water_geometry],
    crs=water_clipped.crs
)

print("Missing-water feature:")
print(missing_water)

print("\nCRS:")
print(missing_water.crs)

print("\nGeometry type:")
print(missing_water.geometry.geom_type.value_counts())

In [ ]:
# Export the missing water area as a GeoPackage

output_path = VECTOR_DIR / "missing_water.gpkg"

missing_water.to_file(
    output_path,
    layer="missing_water",
    driver="GPKG",
)

print("Exported:", output_path)
print("Features:", len(missing_water))
print("CRS:", missing_water.crs)
print("Area (km²):", missing_water["area_km2"].sum())

### 5.5 Quantify and document the remaining soil-data gaps

After clipping the Mayabeque 1:25,000 soil dataset to the Jaruco AOI, the soil polygons covered **162.85 km²** of the **178.47 km²** study area.

The initial uncovered area was **15.62 km²**. Spatial comparison with the water-body dataset showed that **11.42 km²** of this gap corresponds to the portion of the Mamposton reservoir not represented by the soil dataset. This area is therefore treated as a **water/reservoir area rather than missing soil information**.

The remaining **4.20 km²**, approximately **2.35% of the AOI**, consists of small and disconnected areas not covered by the soil dataset and not explained by the mapped reservoir.

For the current GeoSoil-Cuba project, these remaining gaps will **not be investigated or artificially assigned to soil classes**. They will be documented as **unresolved/no-data areas** and reserved for potential future analysis using additional datasets such as satellite imagery or other authoritative geographic data.

Consequently, approximately **97.65% of the AOI is accounted for by the processed soil coverage and identified reservoir area**, providing sufficient spatial coverage for the subsequent analyses.


In [ ]:
# Create the remaining non-water soil-data gaps

remaining_gap_geometry = soil_gap_geometry.difference(water_geometry)

gap_layer = gpd.GeoDataFrame(
    {"gap_id": range(1, len(remaining_gap_geometry.geoms) + 1)},
    geometry=list(remaining_gap_geometry.geoms),
    crs=soil_aoi_clipped.crs,
)

# Validate the remaining gap geometries

print("Features:", len(gap_layer))
print("Empty geometries:", gap_layer.geometry.is_empty.sum())
print("Invalid geometries:", (~gap_layer.geometry.is_valid).sum())
print("CRS:", gap_layer.crs)

# Save the remaining soil-data gaps

gap_output = VECTOR_DIR / "remaining_soil_gaps.gpkg"

gap_layer.to_file(
    gap_output,
    layer="remaining_soil_gaps",
    driver="GPKG",
)

print("Saved:", gap_output)

In [ ]:
# Plot remaining non-water soil-data gaps over the AOI

MAPS_DIR = PROJECT_DIR / "outputs" / "maps"
MAPS_DIR.mkdir(parents=True, exist_ok=True)

output_file = MAPS_DIR / "remaining_soil_data_gaps.png"

fig, ax = plt.subplots(figsize=(10, 8))

aoi_projected.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1.2,
    label="AOI",
)

gap_layer.plot(
    ax=ax,
    alpha=0.7,
    edgecolor="black",
    linewidth=0.5,
    label="Remaining soil-data gaps",
)

ax.set_title("Remaining Non-Water Soil-Data Gaps")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.legend()
ax.set_aspect("equal")

plt.tight_layout()

fig.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Saved: {output_file}")

In [ ]:
figure_path = OUTPUT_DIR / "maps" / "remaining_soil_gaps.png"

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

print("Saved:", figure_path)

## 6. Soil classification and inventory

Let's now examine what soil classes actually occur within the AOI. Before saving the final processed layer, we want a simple inventory of the soil attributes and their spatial distribution.

In [ ]:
# Inspect soil_aoi_clipped columns

print("=== soil_aoi_clipped columns ===\n")

for i, col in enumerate(soil_aoi_clipped.columns, start=1):
    print(f"{i:2}. {col:<25} | dtype: {soil_aoi_clipped[col].dtype}")

print(f"\nTotal columns: {len(soil_aoi_clipped.columns)}")
print(f"Total features: {len(soil_aoi_clipped)}")

### 6.1 Inspect the soil classification attributes

In [ ]:
# Inspect the main soil classification fields

soil_classes = (
    soil_aoi_clipped[
        ["Cód_tipo", "Cód_stipo", "Nombre_subtipo"]
    ]
    .drop_duplicates()
    .sort_values(
        ["Cód_tipo", "Cód_stipo"]
    )
)

print("Unique soil classifications:")
print(soil_classes.to_string(index=False))

print("\nNumber of unique soil types:")
print(soil_aoi_clipped["Cód_tipo"].nunique())

print("\nNumber of unique soil subtypes:")
print(soil_aoi_clipped["Cód_stipo"].nunique())

print("\nNumber of unique named subtypes:")
print(soil_aoi_clipped["Nombre_subtipo"].nunique())

### 6.2 Decode soil type and subtype classifications

The original soil database stores the rectified soil type and subtype codes
in separate fields (`Cód_tipo` and `Cód_stipo`).

According to §2.5.3 and Anexo 6 of the database documentation, the complete
subtype code is constructed by combining the type and subtype codes:

- One-digit type + one-digit subtype: type + `0` + subtype
- One-digit type + two-digit subtype: type + subtype
- Two-digit type + one-digit subtype: type + `0` + subtype
- Two-digit type + two-digit subtype: type + subtype

The corresponding soil type and subtype names are then combined to produce
the full soil-unit name.

For the soil units occurring in the Jaruco AOI, the documented mappings
include:

- 2 + 1 → 201 — Ferralítico Rojo Típico
- 2 + 21 → 221 — Ferralítico Rojo Compactado
- 4 + 1 → 401 — Ferralítico Amarillento Típico
- 4 + 2 → 402 — Ferralítico Amarillento Concrecionario
- 8 + 1 → 801 — Fersialítico Pardo Rojizo Típico
- 10 + 10 → 1010 — Pardo con Carbonatos Negro Grisáceo
- 12 + 12 → 1212 — Húmico Carbonático Gris amarillento
- 15 + 10 → 1510 — Oscuro Plástico Gleyzado Negro Grisáceo
- 13 + 1 → 1301 — Rendzina Rojo Típico
- 17 + 12 → 1712 — Oscuro Plástico no Gleyzado Gris amarillento
- 19 + 2 → 1902 — Gley Ferralítico Concrecionario
- 26 + 18 → 2618 — Aluvial Diferenciado
- 28 + 19 → 2819 — Esquelético Diferenciado

The original database fields are preserved. New fields are added for the
decoded classification so that the source attributes remain traceable.

The `0 + 0` combination is retained as an undefined classification. It is
not assigned a soil type, subtype, or soil name because code `0` is not
defined as a soil type or subtype in the supplied documentation.

In [ ]:
# Documentation-based soil classification lookup

# Soil type names from the documentation
type_names = {
    2: "Ferralítico Rojo",
    4: "Ferralítico Amarillento",
    8: "Fersialítico Pardo Rojizo",
    10: "Pardo con Carbonatos",
    12: "Húmico Carbonático",
    13: "Rendzina Roja",
    15: "Oscuro Plástico Gleyzado",
    17: "Oscuro Plástico no Gleyzado",
    19: "Gley Ferralítico",
    26: "Aluvial",
    28: "Esquelético",
}

# Soil subtype names from the documentation
subtype_names = {
    1: "Típico",
    2: "Concrecionario",
    4: "Hidratado",
    10: "Negro Grisáceo",
    12: "Gris Amarillento",
    18: "Poco Diferenciado",
    19: "Diferenciado",
    21: "Antrópico",
}


def build_complete_subtype_code(type_code, subtype_code):
    """
    Construct the complete subtype code according to §2.5.3.

    Documentation rules:

    1. Type = 1 digit, subtype = 1 digit:
       type + 0 + subtype
       Example: 2 + 1 → 201

    2. Type = 1 digit, subtype = 2 digits:
       type + subtype
       Example: 2 + 21 → 221

    3. Type = 2 digits, subtype = 1 digit:
       type + 0 + subtype
       Example: 10 + 1 → 1001

    4. Type = 2 digits, subtype = 2 digits:
       type + subtype
       Example: 26 + 18 → 2618

     The documented four cases are equivalent to:
        type_code * 100 + subtype_code
    """

    if type_code == 0 and subtype_code == 0:
        return 0

    if type_code < 1 or type_code > 28:
        raise ValueError(f"Invalid soil type code: {type_code}")

    if subtype_code < 1:
        raise ValueError(f"Invalid soil subtype code: {subtype_code}")

    return type_code * 100 + subtype_code


# Build the complete subtype code
soil_codes = []

for type_code, subtype_code in zip(
    soil_aoi_clipped["Cód_tipo"],
    soil_aoi_clipped["Cód_stipo"]
):
    soil_codes.append(
        build_complete_subtype_code(type_code, subtype_code)
    )

soil_aoi_clipped["soil_code"] = soil_codes


# Translate type and subtype codes into names
soil_aoi_clipped["soil_type_name"] = (
    soil_aoi_clipped["Cód_tipo"].map(type_names)
)

soil_aoi_clipped["soil_subtype_name"] = (
    soil_aoi_clipped["Cód_stipo"].map(subtype_names)
)


# Combine type and subtype names
soil_aoi_clipped["soil_name"] = (
    soil_aoi_clipped["soil_type_name"]
    + " "
    + soil_aoi_clipped["soil_subtype_name"]
)


# Keep 0 + 0 explicitly undefined
undefined_mask = (
    (soil_aoi_clipped["Cód_tipo"] == 0)
    & (soil_aoi_clipped["Cód_stipo"] == 0)
)

soil_aoi_clipped.loc[
    undefined_mask,
    [
        "soil_type_name",
        "soil_subtype_name",
        "soil_name",
    ]
] = None


# Inspect the resulting unique classifications
soil_classification = (
    soil_aoi_clipped[
        [
            "Cód_tipo",
            "Cód_stipo",
            "soil_code",
            "soil_type_name",
            "soil_subtype_name",
            "soil_name",
        ]
    ]
    .drop_duplicates()
    .sort_values("soil_code")
)

print(soil_classification.to_string(index=False))

### 6.3 Validate the decoded soil classifications

Before calculating the soil inventory, verify that every non-zero soil polygon
has a documented soil type, subtype, and complete subtype code.

The `0 + 0` feature is intentionally excluded from the missing-name check because
Anexo 6 does not define code 0 as a soil type or subtype.

In [ ]:
# Validate decoded classifications

non_zero = soil_aoi_clipped["Cód_tipo"] != 0

missing_classification = soil_aoi_clipped[
    non_zero
    & (
        soil_aoi_clipped["soil_type_name"].isna()
        | soil_aoi_clipped["soil_subtype_name"].isna()
        | soil_aoi_clipped["soil_name"].isna()
    )
]

print("Non-zero soil polygons:", non_zero.sum())
print("Polygons with missing classification:", len(missing_classification))

if len(missing_classification) > 0:
    print("\nMissing classifications:")
    print(
        missing_classification[
            ["Cód_tipo", "Cód_stipo", "soil_code"]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )
else:
    print("All non-zero soil polygons have a complete classification.")

### 6.4 Calculate the soil inventory by subtype

The clipped soil layer contains 81 classified soil polygons within the AOI.

The inventory will quantify each soil subtype using the clipped geometries:

- Number of polygons
- Area in km²
- Percentage of the total AOI

Area will be calculated after projecting the clipped soil layer to
UTM Zone 17N (EPSG:32617), which provides a metric coordinate system
appropriate for this study area.

The calculation uses the geometry produced by the AOI clip, so polygon
areas represent only the portion of each soil unit that lies inside the AOI.

In [ ]:
# Project clipped soil layer to UTM Zone 17N for area calculations

soil_area = soil_aoi_clipped.to_crs(epsg=32617)

# Calculate polygon area in km²
soil_area["area_km2"] = soil_area.geometry.area / 1_000_000

# Exclude the undefined 0 + 0 feature from the classified soil inventory
classified_soil = soil_area[
    soil_area["Cód_tipo"] != 0
].copy()

# Inventory by complete soil subtype
soil_inventory = (
    classified_soil
    .groupby(
        [
            "soil_code",
            "soil_type_name",
            "soil_subtype_name",
            "soil_name",
        ],
        dropna=False
    )
    .agg(
        polygons=("geometry", "size"),
        area_km2=("area_km2", "sum"),
    )
    .reset_index()
)

# Calculate percentage of the classified soil area
classified_area = soil_inventory["area_km2"].sum()

soil_inventory["percent_classified_area"] = (
    soil_inventory["area_km2"]
    / classified_area
    * 100
)

# Sort by area, largest first
soil_inventory = soil_inventory.sort_values(
    "area_km2",
    ascending=False
).reset_index(drop=True)

print("Classified soil area:", round(classified_area, 3), "km²")
print("\nSoil inventory:")
print(
    soil_inventory.to_string(
        index=False,
        formatters={
            "area_km2": "{:.3f}".format,
            "percent_classified_area": "{:.2f}".format,
        }
    )
)

### 6.5 Inspect undefined soil classifications

In [ ]:
undefined_soil = soil_aoi_clipped[
    (soil_aoi_clipped["Cód_tipo"] == 0)
    & (soil_aoi_clipped["Cód_stipo"] == 0)
]

print("Undefined soil polygons:", len(undefined_soil))

print(
    undefined_soil[
        ["Cód_tipo", "Cód_stipo", "soil_code", "soil_type_name",
         "soil_subtype_name", "soil_name"]
    ].to_string(index=False)
)

### 6.6 Quantify the undefined soil polygon 

One polygon in the clipped soil layer has Cód_tipo = 0 and Cód_stipo = 0. Because Anexo 6 does not define code 0 as a soil type or subtype, it was excluded from the classified soil inventory. Its area will now be calculated separately to determine: 

- its area in km²;
- its percentage of the total AOI;
- the relationship between classified and undefined areas.

This allows us to distinguish the total AOI area from the area represented by classified soil units.

In [ ]:
# Calculate the area of the undefined 0 + 0 polygon

undefined_soil = soil_area[
    (soil_area["Cód_tipo"] == 0)
    & (soil_area["Cód_stipo"] == 0)
].copy()

undefined_area_km2 = undefined_soil["area_km2"].sum()

aoi_area_km2 = soil_area.union_all().area / 1_000_000

print("AOI area represented by soil layer:", round(aoi_area_km2, 3), "km²")
print("Classified soil area:", round(classified_area, 3), "km²")
print("Undefined 0 + 0 area:", round(undefined_area_km2, 3), "km²")

print(
    "Undefined area as % of AOI:",
    round(undefined_area_km2 / aoi_area_km2 * 100, 2),
    "%"
)

print(
    "Classified area as % of AOI:",
    round(classified_area / aoi_area_km2 * 100, 2),
    "%"
)

print(
    "Classified + undefined:",
    round(classified_area + undefined_area_km2, 3),
    "km²"
)

## 7. Soil topology and area-accounting validation

The initial area calculation reveals a small difference between the sum of
individual clipped polygon areas and the area of their geometric union.

- Sum of classified soil areas: 161.899 km²
- Undefined 0 + 0 polygon: 0.922 km²
- Sum of all polygon areas: 162.821 km²
- Union area: 162.766 km²

The difference is 0.055 km².

Before treating the soil inventory as final, we need to determine whether
this difference is caused by overlapping soil polygons, geometry artifacts,
or another feature of the clipped dataset.

In [ ]:
# Validate overlaps and calculate the area using union_all()

# Total area of individual clipped polygons
polygon_area_total = soil_area.geometry.area.sum() / 1_000_000

# Area of the geometric union
union_area_km2 = (
    soil_area.geometry.union_all().area / 1_000_000
)

difference_km2 = polygon_area_total - union_area_km2

print("Sum of individual polygon areas:",
      round(polygon_area_total, 3), "km²")

print("Union area:",
      round(union_area_km2, 3), "km²")

print("Difference:",
      round(difference_km2, 3), "km²")

### 7.1 Identify polygons responsible for overlap

In [ ]:
# Check for overlapping clipped soil polygons

overlap_pairs = []

geometries = soil_area.geometry

for i in range(len(soil_area)):
    for j in range(i + 1, len(soil_area)):
        if geometries.iloc[i].overlaps(geometries.iloc[j]):
            overlap_pairs.append((i, j))

print("Number of overlapping polygon pairs:", len(overlap_pairs))

if overlap_pairs:
    print("\nOverlapping pairs:")
    
    for i, j in overlap_pairs:
        row_i = soil_area.iloc[i]
        row_j = soil_area.iloc[j]

        print(
            f"\nPolygon indices: {i} and {j}"
        )
        print(
            f"  {row_i['soil_code']} - {row_i['soil_name']}"
        )
        print(
            f"  {row_j['soil_code']} - {row_j['soil_name']}"
        )

### 7.2 Quantify pairwise overlap areas

The clipped soil layer contains 90 pairs of polygons whose interiors overlap.
The number of pairs alone does not indicate the amount of overlapping area,
because multiple polygons may overlap the same location.

We therefore calculate the area of every pairwise intersection and summarize
the total pairwise overlap.

This is a diagnostic step only. No geometries will be modified or removed
until the magnitude and pattern of the overlaps are understood.

In [ ]:
# Quantify pairwise overlap areas

import pandas as pd

overlap_details = []

geometries = soil_area.geometry

for i in range(len(soil_area)):
    for j in range(i + 1, len(soil_area)):
        
        intersection = geometries.iloc[i].intersection(
            geometries.iloc[j]
        )
        
        if not intersection.is_empty:
            area_km2 = intersection.area / 1_000_000
            
            if area_km2 > 0:
                overlap_details.append({
                    "index_1": i,
                    "index_2": j,
                    "soil_code_1": soil_area.iloc[i]["soil_code"],
                    "soil_code_2": soil_area.iloc[j]["soil_code"],
                    "soil_name_1": soil_area.iloc[i]["soil_name"],
                    "soil_name_2": soil_area.iloc[j]["soil_name"],
                    "overlap_km2": area_km2,
                })

overlap_df = pd.DataFrame(overlap_details)

print("Polygon pairs with positive-area overlap:",
      len(overlap_df))

print(
    "Total pairwise overlap area:",
    round(overlap_df["overlap_km2"].sum(), 6),
    "km²"
)

print(
    "\nLargest overlaps:"
)

print(
    overlap_df
    .sort_values("overlap_km2", ascending=False)
    .head(20)
    .to_string(
        index=False,
        formatters={
            "overlap_km2": "{:.6f}".format
        }
    )
)

### 7.3 Summarize the topology check

90 polygon pairs showed positive-area overlaps totaling 0.054919 km². The largest overlap (0.046807 km²) occurs between Aluvial Poco Diferenciado (2618) and Pardo con Carbonatos Típico (1001). The overlap represents a negligible proportion of the mapped soil area and will be investigated during a later spatial/topological analysis. Source geometries are retained unchanged.

## 8. Export and quality-control outputs

The final soil layer is prepared for use in subsequent notebooks and GIS applications. It contains the soil classification name, the area of each soil feature, and its geometry.

The original Spanish soil names are retained, and tentative English translations are added in a separate field to support the international presentation of GeoSoil-Cuba. These translations are descriptive and do not represent official equivalences with international soil classification systems.

The resulting GeoPackage contains three layers:

- soils: soil polygons clipped to the Jaruco AOI, with their classification names and areas.
- missing_water: the area identified as water-related soil-map coverage gaps.
- remaining_soil_gaps: the remaining uncovered areas not explained by the water layer.

The GeoPackage is then checked to confirm that the expected soil attributes have been exported.



In [ ]:
# Prepare the analysis-ready soil layer

soil_name_translations = {
    "Rendzina Roja Típico":
        "Typical Red Rendzina",
    "Oscuro Plástico no Gleyzado Gris Amarillento":
        "Yellowish-Gray Non-Gleyed Dark Plastic Soil",
    "Ferralítico Rojo Típico":
        "Typical Red Ferrallitic Soil",
    "Ferralítico Rojo Compactado":
        "Compacted Red Ferrallitic Soil",
    "Ferralítico Rojo Hidratado":
        "Hydrated Red Ferrallitic Soil",
    "Ferralítico Rojo Antrópico":
        "Anthropic Red Ferrallitic Soil",
    "Ferralítico Amarillento Típico":
        "Typical Yellowish Ferrallitic Soil",
    "Ferralítico Amarillento Concrecionario":
        "Concretionary Yellowish Ferrallitic Soil",
    "Fersialítico Pardo Rojizo Típico":
        "Typical Reddish-Brown Fersialitic Soil",
    "Pardo con Carbonatos Negro Grisáceo":
        "Grayish-Black Brown Soil with Carbonates",
    "Pardo con Carbonatos Típico":
        "Typical Brown Soil with Carbonates",
    "Húmico Carbonático Gris amarillento":
        "Yellowish-Gray Humic Carbonatic Soil",
    "Húmico Carbonático Típico":
        "Typical Humic Carbonatic Soil",
    "Oscuro Plástico Gleyzado Negro Grisáceo":
        "Grayish-Black Gleyed Dark Plastic Soil",
    "Gley Ferralítico Concrecionario":
        "Concretionary Ferrallitic Gley Soil",
    "Aluvial Diferenciado":
        "Differentiated Alluvial Soil",
    "Aluvial Poco Diferenciado":
        "Weakly Differentiated Alluvial Soil",
    "Esquelético Diferenciado":
        "Differentiated Skeletal Soil",
}

# Select the attributes required for the final soil layer
soil_aoi_ready = soil_aoi_clipped[
    [
        "soil_name",
        "geometry",
        "area_km2",
    ]
].copy()

# Add tentative English translations
soil_aoi_ready["soil_name_en"] = (
    soil_aoi_ready["soil_name"]
    .map(soil_name_translations)
)

# Check that all named soil classes have translations
untranslated_classes = (
    soil_aoi_ready.loc[
        soil_aoi_ready["soil_name"].notna()
        & soil_aoi_ready["soil_name_en"].isna(),
        "soil_name",
    ]
    .unique()
)

if len(untranslated_classes) > 0:
    print("WARNING: Soil classes without English translations:")
    for name in untranslated_classes:
        print(f"  - {name}")
else:
    print("All named soil classes have English translations.")

# Export the analysis-ready GeoPackage
output_path = VECTOR_DIR / "soil_aoi_ready.gpkg"

# Soil layer
soil_aoi_ready.to_file(
    output_path,
    layer="soils",
    driver="GPKG",
)

# Missing-water layer
missing_water.to_file(
    output_path,
    layer="missing_water",
    driver="GPKG",
)

# Remaining soil-data gaps
gap_layer.to_file(
    output_path,
    layer="remaining_soil_gaps",
    driver="GPKG",
)

print("\nExported:", output_path)
print("Soil features:", len(soil_aoi_ready))
print("CRS:", soil_aoi_ready.crs)

print("\nSoil layer columns:")
for i, col in enumerate(soil_aoi_ready.columns, start=1):
    print(f"{i:2}. {col}")

### 8.1 Visualize the analysis-ready soil units

The soil classes are displayed using cartographic colors inspired by their documented soil names. These colors are visualization choices, not measured soil-color values.


In [ ]:
# Plot soil units using soil-type-inspired natural colors

soil_colors_es = {
    # Ferralítico Rojo
    "Ferralítico Rojo Típico": "#A63D2F",
    "Ferralítico Rojo Compactado": "#8F2F24",
    "Ferralítico Rojo Hidratado": "#B85A3A",
    "Ferralítico Rojo Antrópico": "#7F3028",

    # Ferralítico Amarillento
    "Ferralítico Amarillento Típico": "#C9A227",
    "Ferralítico Amarillento Concrecionario": "#A98217",

    # Fersialítico
    "Fersialítico Pardo Rojizo Típico": "#8B5A3C",

    # Pardo con Carbonatos
    "Pardo con Carbonatos Negro Grisáceo": "#5E5A50",
    "Pardo con Carbonatos Típico": "#806A4A",

    # Húmico Carbonático
    "Húmico Carbonático Gris amarillento": "#9A956F",
    "Húmico Carbonático Típico": "#756B4A",

    # Oscuro Plástico
    "Oscuro Plástico Gleyzado Negro Grisáceo": "#454545",
    "Oscuro Plástico no Gleyzado Gris Amarillento": "#77735C",

    # Rendzina
    "Rendzina Roja Típico": "#9B4935",


    # Gley Ferralítico
    "Gley Ferralítico Concrecionario": "#6B6045",

    # Aluvial
    "Aluvial Diferenciado": "#B58B5A",
    "Aluvial Poco Diferenciado": "#C7A77A",

    # Esquelético
    "Esquelético Diferenciado": "#C4B69A",
}



In [ ]:
# Plot soil units using soil-type-inspired natural colors

soil_colors = {
    # Red Ferrallitic Soil
    "Typical Red Ferrallitic Soil": "#A63D2F",
    "Compacted Red Ferrallitic Soil": "#8F2F24",
    "Hydrated Red Ferrallitic Soil": "#B85A3A",
    "Anthropic Red Ferrallitic Soil": "#7F3028",

    # Yellowish Ferrallitic Soil
    "Typical Yellowish Ferrallitic Soil": "#C9A227",
    "Concretionary Yellowish Ferrallitic Soil": "#A98217",

    # Reddish-Brown Fersialitic Soil
    "Typical Reddish-Brown Fersialitic Soil": "#8B5A3C",

    # Brown Soil with Carbonates
    "Grayish-Black Brown Soil with Carbonates": "#5E5A50",
    "Typical Brown Soil with Carbonates": "#806A4A",

    # Humic Carbonatic Soil
    "Yellowish-Gray Humic Carbonatic Soil": "#9A956F",
    "Typical Humic Carbonatic Soil": "#756B4A",

    # Dark Plastic Soil
    "Grayish-Black Gleyed Dark Plastic Soil": "#454545",
    "Yellowish-Gray Non-Gleyed Dark Plastic Soil": "#77735C",

    # Red Rendzina
    "Typical Red Rendzina": "#9B4935",

    # Ferrallitic Gley Soil
    "Concretionary Ferrallitic Gley Soil": "#6B6045",

    # Alluvial Soil
    "Differentiated Alluvial Soil": "#B58B5A",
    "Weakly Differentiated Alluvial Soil": "#C7A77A",

    # Skeletal Soil
    "Differentiated Skeletal Soil": "#C4B69A",
}



In [ ]:
plot_colors = soil_aoi_ready["soil_name"].map(soil_colors)

missing_colors = (
    soil_aoi_ready.loc[plot_colors.isna(), "soil_name"]
    .dropna()
    .unique()
)

print("Soil classes without assigned colors:")

if len(missing_colors) == 0:
    print("None")
else:
    for soil in missing_colors:
        print(f"  - {soil}")

In [ ]:
from matplotlib.patches import Patch

MAPS_DIR = PROJECT_DIR / "outputs" / "maps"
MAPS_DIR.mkdir(parents=True, exist_ok=True)

output_file = MAPS_DIR / "soil_units_aoi.png"

# Assign colors to soil classes.
# Undefined classifications are displayed in neutral gray.
plot_colors = (
    soil_aoi_ready["soil_name_en"]
    .map(soil_colors)
    .fillna("#BDBDBD")
)

fig, ax = plt.subplots(figsize=(14, 10))

# AOI boundary
aoi_projected.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1.2,
)

# Soil units
soil_aoi_ready.plot(
    ax=ax,
    color=plot_colors,
    edgecolor="black",
    linewidth=0.4,
)

# Water-related soil-data gap
missing_water.plot(
    ax=ax,
    color="#4A90E2",
    edgecolor="#1F4E79",
    linewidth=0.8,
)

# Remaining soil-data gaps
gap_layer.plot(
    ax=ax,
    color="#D9D9D9",
    edgecolor="#777777",
    linewidth=0.5,
)

# Build soil legend
legend_handles = [
    Patch(
        facecolor=color,
        edgecolor="black",
        label=soil,
    )
    for soil, color in soil_colors.items()
]

# Add undefined classification when present
if soil_aoi_ready["soil_name"].isna().any():
    legend_handles.append(
        Patch(
            facecolor="#BDBDBD",
            edgecolor="black",
            label="Undefined classification",
        )
    )

# Add water and remaining-gap legend entries
legend_handles.extend(
    [
        Patch(
            facecolor="#4A90E2",
            edgecolor="#1F4E79",
            label="Water-related gap",
        ),
        Patch(
            facecolor="#D9D9D9",
            edgecolor="#777777",
            label="Remaining soil-data gaps",
        ),
    ]
)

ax.legend(
    handles=legend_handles,
    title="Soil units and data gaps",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=True,
    fontsize=9,
    title_fontsize=10,
)

ax.set_title("Soil Units and Data Gaps within the GeoSoil-Cuba AOI")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_aspect("equal")

plt.tight_layout()

fig.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Saved: {output_file}")


### 8.2 Verify the analysis-ready GeoPackage

The exported GeoPackage is inspected to confirm its layers, feature counts, CRS, and attributes before the processed tables are exported.


In [ ]:
# Inspect the layers and columns in the analysis-ready GeoPackage

import fiona
import geopandas as gpd

output_path = VECTOR_DIR / "soil_aoi_ready.gpkg"

# List all layers
layers = fiona.listlayers(output_path)

print("Layers:")
for layer in layers:
    print(f"  - {layer}")

# Inspect each layer
for layer in layers:
    gdf = gpd.read_file(output_path, layer=layer)

    print(f"\n{'=' * 60}")
    print(f"Layer: {layer}")
    print(f"{'=' * 60}")

    print(f"Features: {len(gdf):,}")
    print(f"CRS: {gdf.crs}")

    print("\nColumns:")
    for i, column in enumerate(gdf.columns, start=1):
        print(f"  {i:2}. {column}")

    print("\nGeometry types:")
    print(gdf.geometry.geom_type.value_counts())

In [ ]:
# Inspect the first 10 rows of the soil_aoi_clipped layer


soil_aoi_ready.head(100)

### 8.3 Export the soil classification lookup 

This preserves the relationship between the original type/subtype codes, the complete soil code, and their decoded names.

In [ ]:
# Add tentative English soil names
soil_aoi_clipped["soil_name_en"] = (
    soil_aoi_clipped["soil_name"].map(soil_name_translations)
)

# ---------------------------------------------------------
# Soil classification lookup
# ---------------------------------------------------------

classification_lookup = (
    soil_aoi_clipped[
        [
            "Cód_tipo",
            "Cód_stipo",
            "soil_code",
            "soil_type_name",
            "soil_subtype_name",
            "soil_name",
            "soil_name_en",
        ]
    ]
    .drop_duplicates()
    .sort_values("soil_code")
    .reset_index(drop=True)
)

# Rename the original Spanish name explicitly
classification_lookup = classification_lookup.rename(
    columns={
        "soil_name": "soil_name_es",
    }
)

lookup_path = TABLE_DIR / "soil_classification_lookup.csv"

classification_lookup.to_csv(
    lookup_path,
    index=False,
    encoding="utf-8-sig",
)

print("Exported:", lookup_path)
print("Lookup records:", len(classification_lookup))





### 8.4 Export the classified soil inventory

The table records the number of polygons, mapped area, and percentage of classified soil area for each soil type/subtype.

In [ ]:
# ---------------------------------------------------------
# Soil inventory
# ---------------------------------------------------------

soil_inventory = (
    soil_area[soil_area["soil_code"] != 0]
    .groupby(
        ["soil_code", "soil_name"],
        dropna=False,
    )
    .agg(
        polygons=("soil_code", "size"),
        area_km2=("area_km2", "sum"),
    )
    .reset_index()
)

# Add tentative English translations
soil_inventory["soil_name_en"] = (
    soil_inventory["soil_name"].map(soil_name_translations)
)

# Rename the original Spanish name explicitly
soil_inventory = soil_inventory.rename(
    columns={
        "soil_name": "soil_name_es",
    }
)

soil_inventory["percent_classified_area"] = (
    soil_inventory["area_km2"]
    / soil_inventory["area_km2"].sum()
    * 100
)

soil_inventory = (
    soil_inventory
    .sort_values("area_km2", ascending=False)
    .reset_index(drop=True)
)

inventory_path = TABLE_DIR / "soil_inventory.csv"

soil_inventory.to_csv(
    inventory_path,
    index=False,
    encoding="utf-8-sig",
)

print("Exported:", inventory_path)
print("Soil classes:", len(soil_inventory))
print(
    "Classified soil area:",
    round(soil_inventory["area_km2"].sum(), 3),
    "km²",
)
print(
    "Percentage total:",
    round(soil_inventory["percent_classified_area"].sum(), 3),
    "%",
)

print("\nInventory:")
print(soil_inventory.to_string(index=False))


# Workflow summary

This notebook prepares the Mayabeque 1:25,000 soil map for spatial analysis
within the GeoSoil-Cuba Jaruco/Mayabeque AOI.

### 1 Project setup and source data

- Loaded the Mayabeque soil map and Jaruco AOI.
- Verified the source datasets and their coordinate reference systems.
- Identified the soil map as **NAD27 / Cuba Norte (EPSG:3795)**.
- Reprojected the AOI to the soil-map working CRS while preserving the original datasets.

### 2 Geometry validation and repair

- Checked all source soil geometries for validity.
- Identified and diagnosed invalid geometries.
- Repaired invalid geometries with `make_valid()`.
- Normalized repaired `GeometryCollection` objects to polygonal geometries.
- Confirmed the analysis soil layer contains **1,201 features with no null or invalid geometries**.

### 3 AOI intersection, clipping, and soil coverage

- Identified the soil features intersecting the AOI.
- Clipped the candidate soil features to the study area.
- Calculated individual polygon areas in a projected CRS.
- Determined the spatial union of the clipped soil layer.
- Established approximately **162.85 km² of soil spatial coverage within the 178.47 km² AOI**.

### 4 Water-body analysis

- Loaded and inspected the water-body dataset.
- Identified the water features relevant to the AOI.
- Clipped the relevant water geometry to the AOI.
- Validated the resulting water geometries and calculated their area.
- Used the water dataset to distinguish reservoir area from apparent soil-data gaps.

### 5 Soil-data gaps and reservoir exclusion

- Calculated the initial soil-data gap.
- Compared the gap with the Mampostón reservoir.
- Created the portion of the reservoir not represented by the soil map as `missing_water`.
- Identified approximately **4.20 km² of remaining non-water soil-data gaps**, representing about **2.35% of the AOI**.
- Retained these areas as unresolved/no-data areas rather than assigning unsupported soil classes.

### 6 Soil classification and inventory

- Inspected the source soil classification attributes.
- Decoded soil type and subtype codes using the supplied documentation.
- Preserved the original classification fields for traceability.
- Validated the decoded classifications.
- Calculated the classified soil inventory.
- Treated the `0 + 0` classification as undefined because the supplied documentation does not define code `0` as a soil type or subtype.

### 7 Soil topology and area-accounting validation

- Compared the sum of individual polygon areas with the spatial union.
- Identified **90 polygon pairs with positive-area overlaps**.
- Quantified the total pairwise overlap at approximately **0.054919 km²**.
- Retained the source-derived geometries rather than removing or altering overlapping polygons without sufficient evidence for a correction.

### 8 Export and quality control

The notebook produces the processed spatial and tabular outputs required by the next stages of GeoSoil-Cuba:

- `soil_aoi_ready.gpkg`
- `missing_water`
- `remaining_soil_gaps`
- `soil_classification_lookup.csv`
- `soil_inventory.csv`
- Soil and gap map figures in `outputs/maps/`




# Stage 1 conclusions

The Mayabeque 1:25,000 soil dataset has been spatially prepared, classified,
validated, and documented for subsequent GeoSoil-Cuba analysis.

